# Introduction

<div style="
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 12px 16px;
    max-width: 500px;
    background-color: #f9f9f9;
">
  <strong>✈️⚡ Electric Aircraft Design</strong><br>
  <br>
  <strong>Author:</strong> Florent LUTZ<br>
  <strong>Affiliation:</strong> ISAE-SUPAERO / DCAS <br>
  <strong>Course:</strong> Electric Aircraft Design<br>
</div>

## Overview of the course

This course is designed to help you gain hands-on experience with the design of aircraft with electrified propulsive architecture. It will show you the pros and cons of different electrification scenarios and highlight a few of the design drivers that dictate the sizing of the powertrain. It is structured in two parts that will use two aircraft platform as use cases: the Quest/DAHER Kodiak 100 and the Pipistrel Velis Electro/Club. 

1. [**Study of sensitivity to design choices**](./01_Sensitivity_studies.ipynb) <br>
In this application, you will explore how some of the design choices that can be made for electrified powertrain affect the aircraft it is installed in. 

2. [**Comparaison of the design of multiple powertrains**](./02_Configurations_comparison.ipynb) <br>
In this application, you will compare the design of multiples aircraft with the same requirement but with different propulsive architectures. 

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
This course is designed to be understood and ran with minimal knowledge in Python. Basic knowledge, however is helpful. The user will be mostly expected to input some values corresponding to the exercise and execute the cells. Some cells however should not be changed and require execution as is for the lab to work correctly. They will be marked accordingly.
</div>

<div style="
  border-left: 4px solid #2e7d32;
  padding: 10px 12px;
  background-color: #f3faf4;
  margin: 10px 0;
">
<strong>📚 Additional Resources</strong><br>
This course, presented in the form of interactive <a href="https://jupyter.org/">Jupyter Notebooks</a>, makes use of the following software and resources:
<br><br>
<strong>Software</strong>: Most of the calculations and analysis carried out in the Notebooks rely on the open-source <a href="https://github.com/florentLutz/FAST-OAD-CS23-HE">FAST-OAD-CS23-HE</a> package. The FAST-OAD-CS23-HE framework extends the existing FAST-OAD-GA platform, focused on General Aviation aircraft, to enable medium-fidelity sizing of hybrid-electric aircraft through component models dependent on operating conditions and sizing criteria. It builds upon the Overall Aircraft Design methodologies originally developed by ONERA and ISAE-SUPAERO and incorporates a library of models that physically represent the hybrid-electric powertrain components using simulation and estimation models. Furthermore, it allows for the dynamic formulation of the MDAO problem based on a user input architecture for the powertrain.
<br><br>
Again, no prior knowledge of these software is required as all the preparation work will done in advance. The Jupyter Notebooks contain pre-defined functions and code snippets, allowing you to focus on exploring electric aircraft design concepts and conducting exercises without worrying about complex software manipulations. For more information about the packages or to use the full version, the user is encouraged to check out the <a href="https://fast-oad-cs23-he.readthedocs.io/en/latest/">code documentation</a> and <a href="https://github.com/florentLutz/FAST-OAD-CS23-HE">GitHub repository</a>.
</div>

## Overview of the first use case

The Kodiak 100 is a single engine turboprop utility aircraft with a high-wing and conventional tail configuration. It is versatile aircraft powered by a PT6A-34 engine. Combined with a four-bladed propeller, the aircraft has Short Take Off and Landing capabilities and can carry payload of more than 1100 kg. While the Kodiak is not the most common airframe certified under the CS23 category, it remains one of the most manufactured airframe and has other characteristics which make it a suitable candidate. For one, it has similar performances as other aircraft used as test benches such as the Cessna Grand Caravan modified by Ampaire or the full electric De Havilland Beaver operated by the Harbour Air. Additionally, the aircraft has a very versatile cabin with lots of room to accommodate large batteries and flies at relatively low speeds. It also flies at low altitude removing the need to pressurize the cabin.

<p align="center">
  <img src="./resources/kodiak_100.jpg" width="1200">
  <br>
  <em>Figure 1 : Kodiak 100 — Credits: <a href="https://kodiak.aero/kodiak/">DAHER</a></em>
</p>

An investigation into the most common use of the Kodiak 100 reveals that, while the aircraft can theoretically achieve range of more than 1000nm, it is most commonly used on much shorter ranges, making it a suitable candidate for electrification. The following pictures illustrates the repartition of Kodiak 100 flights reported in the [OpenSky network](https://opensky-network.org/) database. The OpenSky network is a community based receiver network which continuously collect data for air traffic research. Through this network, various database, including one containing flight data, are made available.

<p align="center">
  <img src="./resources/kodiak_100_flight_distribution.svg" width="1200">
  <br>
  <em>Figure 2 : Distribution of Kodiak 100 flights based on the range</em>
</p>

# Baseline aircraft

As a way to familiarize yourself with the code and with the post-processing functions it offers, the reference Kodiak 100 will be computed here.  It will also be used as the reference baseline against which the electrified designs will be compared. As a reminder, the Kodiak 100 is turboprop powered aircraft, so its propulsive architecture relies purely on thermal power. 

Let's start by defining the path to a few directories that will be used in the process.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# A directory to store the files descibing the processes, the powertrain desciption 
# files and the input file.
DATA_FOLDER_PATH = "data"

# A directory that will contain all the intermediate computations.
WORK_FOLDER_PATH = "workdir"

# A directory that will contain the files at the output of the project.
RESULT_FOLDER_PATH = "results"

## Propulsive architecure

In FAST-OAD-CS23-HE, the propulsive architecture is an input of the design process that can easily be changed by the user in a [powertrain description file](https://fast-oad-cs23-he.readthedocs.io/en/latest/documentation/powertrain_builder/pt_file.html).  Any architecture can be evaluated as long as it is built with the components available in the model library. The first step of the design process is thus to transform the actual propulsive architecture of the Kodiak 100 in an architecture modeled based on the available components. This architecture is then translated into the powertrain description file by listing the components and how they are connected. In the case of the Kodiak 100, this translates into a simple powertrain consisting of a propeller, a turboshaft, some fuel lines and two tanks (one in each wing). The corresponding powertrain description file is available [here](./data/kodiak_100_propulsive_architecture.yml) in its native format (.YML). It can also be displayed using FAST-OAD-CS23-HE's post-processing function.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
import pathlib
from fastga_he.gui.power_train_network_viewer import power_train_network_viewer
from IPython.display import IFrame

# Define path to the required files
path_to_powertrain_description_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_propulsive_architecture.yml"
path_to_network_viewer_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_network_viewer.html"

# Generate the network viewer file
power_train_network_viewer(path_to_powertrain_description_file, path_to_network_viewer_file, orientation="LR", legend_position="TL")

# Display the network viewer file
IFrame(src=path_to_network_viewer_file, width="110%", height="500px")

## Design parameters

The aircraft will be sized on a design mission of 1144 nautical miles with four passenger. For the other inputs, they are either extracted from Jane's All the World's Aircraft (available at the school's library), from commercial materials or they are taken from the aircraft's Pilot Operating Handbook. A summary of the main inputs is given in the table below.

| Parameter                                     | Value    |
|-----------------------------------------------|----------| 
| Number of passengers on the design mission    | 4        |
| Design range                                  | 1144 nm  |
| Design mission cruise speed                   | 120 KTAS |
| Design mission cruise altitude                | 10000 ft |
| Approach speed                                | 78 KTAS  |
| Wing aspect ratio                             | 8.41     |
| Engine rated power                            | 551 kW   |

In the FAST-OAD framework (FAST-OAD-CS23-HE being a plugin of the FAST-OAD framework), inputs are handled in the [input file](https://fast-oad.readthedocs.io/en/latest/documentation/usage.html#how-to-generate-an-input-file) which uses the .XML format. Because of the richness of the models in the FAST-OAD-CS23-HE framework, input files can contain hundreds of variables. While the naming convention helps indentifying them by listing the aircraft design disciplines and components they are related to, not all will be discussed here. Some variables describe choices on the aircraft architecture, like the shape of the empennage or the landing gear configuration, some describe the TLARs, etc ... In the end, only a few tens of inputs are really worth investigating, such as the hybridization ratio or the wing geometry. For this laboratory session, the inputs that will need to be changed will be highlighted.

The input file for the case of the Kodiak 100 is available [here](./data/kodiak_100_input.xml) or can be viewed using FAST-OAD post-processing features by executing the next cell.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
When using the variable viewer in the cell below, you can filter the variables in the input file by types (input/output), disciplines (geometry, aerodynamics, ...), components (wing, fuselage, ...). The variable viewer relies on the variable name to perform the filtering, therefore, a variable's value can be accessed using its name. 
    
For instance, a variable named <i>"data:geometry:wing:area"</i> (representing the area of the wing), can be accessed by setting a first filter to <i>data</i>, then a second one to <i>geometry</i>, then a third on to <i>wing</i> and finally a last one to <i>area</i>. Any number of filters can be added depending on the number of categories in the variable name. The pictures below explain the procedure.
</div>

<p align="center">
  <img src="./resources/variable_viewer_process.png" width="1200">
</p>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
import fastoad.api as oad

# Define path to the required files
path_to_input_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_input.xml"

oad.variable_viewer(path_to_input_file)

## Design process definition

Lastly, the configuration file must be defined. The [configuration file](https://fast-oad.readthedocs.io/en/v1.8.4/documentation/usage.html#configuration-file) is, in the FAST-OAD framework, the heart of the process. It allows for the customization of the design process by letting the user change the modules that are run. In that first execrise, we will use a configuration file that corresponds to a design from scratch. The configuration file that we will use for the Kodiak 100 design process is available [here](./data/kodiak_100_input.xml). By executing the next cell, you can generate a N2 diagram (more convenient to understand the process and intercations between modules) corresponding to that design process. It might take a couple of minutes.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
%%capture
# This hides the verbose FAST-OAD module management

# Import useful packages
import warnings

# Define path to the required files
path_to_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_configuration.yml"
path_to_n2_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_n2.html"

# Write the N2 file, removing all useless warning
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    oad.write_n2(path_to_configuration_file, path_to_n2_file, overwrite=True)

Due to some compatibility issues between Jupyter and the N2 generated by OpenMDAO, it needs to be opened in a separate tab. You can do that by going inside laboratories/results, and then right click on the file named kodiak_100_n2.html and click on "Open in New Browser Tab"

## Running the process

Now that all the paths to the files of interest have been defined, the process can be launched. It will be done using the next cell. We start by generating the final input file (FAST-OAD filters out useless inputs automatically, meaning all variables inside this final input file will be used). Then we use FAST-OAD to run the design process. At the end of the process, the results will be automatically stored in an output file which will then be used in post-processing functions or for the analysis.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Generate the final list of inputs, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    oad.generate_inputs(path_to_configuration_file, path_to_input_file, overwrite=True)

# Run the design process, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_design_process = oad.evaluate_problem(path_to_configuration_file, overwrite=True)

## Post-processing

We can now compare the results of the sizing process with actual values taken from literature or from the manufacturer's website. To access the values of the variables of interest in FAST-OAD, we simply require their name.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
OpenMDAO, which the FAST-OAD framework builds upon, supports an automatic conversion between units. To ensure we compare the quantites with the right units, we will do the conversion.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
from tabulate import tabulate
from IPython.display import HTML, display
import openmdao.api as om

# Access the results with a datafile, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_datafile = oad.DataFile(kodiak_100_design_process.output_file_path)

# Print the converted results with the actual value
cell_list = [
    ["Parameter", "Unit", "Computed value", "Actual value"], 
    ["MTOW", "kg", om.convert_units(kodiak_100_datafile["data:weight:aircraft:MTOW"].value[0], kodiak_100_datafile["data:weight:aircraft:MTOW"].units, "kg"), 3290],
    ["OWE", "kg", om.convert_units(kodiak_100_datafile["data:weight:aircraft:OWE"].value[0], kodiak_100_datafile["data:weight:aircraft:OWE"].units, "kg"), 1712],
    ["Design mission fuel", "kg", om.convert_units(kodiak_100_datafile["data:mission:sizing:fuel"].value[0], kodiak_100_datafile["data:mission:sizing:fuel"].units, "kg"), 960],
    ["Wing area", "m**2", om.convert_units(kodiak_100_datafile["data:geometry:wing:area"].value[0], kodiak_100_datafile["data:geometry:wing:area"].units, "m**2"), 22.3],
    ["HTP area", "m**2", om.convert_units(kodiak_100_datafile["data:geometry:horizontal_tail:area"].value[0], kodiak_100_datafile["data:geometry:horizontal_tail:area"].units, "m**2"), 6.21],
    ["VTP area", "m**2", om.convert_units(kodiak_100_datafile["data:geometry:vertical_tail:area"].value[0], kodiak_100_datafile["data:geometry:vertical_tail:area"].units, "m**2"), 2.92],
]

display(HTML(tabulate(cell_list, tablefmt="html")))

Now that the process is done running, post-processing functions can also be used to analyse the mass breakdown of the aircraft, its behaviour during the design mission, ... You can also use the variable viewer to look at the output file and have the values of all parameters computed by the code.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
Remember, you can filter variables in the variable viewer.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
from fastga.utils.postprocessing.analysis_and_plots import mass_breakdown_bar_plot

# Shows all the variable in an interactive table
oad.variable_viewer(kodiak_100_design_process.output_file_path)

# Shows the masses of the aircraft
mass_breakdown_bar_plot(kodiak_100_design_process.output_file_path)

In [ ]:
# Import useful packages
from fastga_he.gui.power_train_weight_breakdown import power_train_mass_breakdown

# Shows the masses of the powertrain
power_train_mass_breakdown(kodiak_100_design_process.output_file_path, path_to_powertrain_description_file)

In [ ]:
# Import useful functions
from fastga_he.gui.performances_viewer import PerformancesViewer

# Define the path to the files containing mission data
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_propulsion_data.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_mission_file.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

## Overview of the second use case

For the second use case, we will use aircraft from the Slovenian manufacturer Pipistrel. We won't go here into as much details as for the Kodiak because it is the purpose of [**the second application Notebook**](./02_Configurations_comparison.ipynb) to compare those designs. They will however be briefly introduced here. These aircraft - the Pipistrel Virus SW 121 and the Pipistrel Velis Electro, shown in the Figure below - were selected because they utilize the same airframe but they have different powertrains. They thus provide a verifiable comparaison of the pros and cons of electric propulsion.

<p align="center">
  <img src="./resources/hero-pipistrel-both.jpg" width="1200">
  <br>
  <em>Figure 3 : the Pipistrel Virus SW 121 and the Pipistrel Velis Electro. Credits: <a href="https://www.pipistrel-aircraft.com/">Pipistrel</a></em>
</p>